In [1]:
#imports 
import re
import unicodedata
import numpy as np
import pandas as pd
from langchain_openai import ChatOpenAI ,OpenAIEmbeddings
from langchain_openrouter import ChatOpenRouter
from dotenv import load_dotenv
from typing import List, Any
import json 
load_dotenv()

True

In [2]:
from pathlib import Path
import sys
import pandas as pd





BASELINE_DATASET_TEMP_0_7_PATH = "./datasets/"+"math_baseline_temp_0.7.csv"
BASELINE_DATASET_TEMP_0_PATH = "./datasets/"+"math_baseline_temp_0.csv"
EVOLUTION_DATASET_PATH = "./datasets/"+"math_evol.csv"
PERSONA_GENERAL_DATASET_PATH = "./datasets/"+"math_persona_general.csv"
PERSONA_MATH_DATASET_PATH = "./datasets/"+"math_persona.csv"
SELF_INSTRUCT_DATASET_PATH = "./datasets/"+"math_self_instruct.csv"



baselineDf_0_7 = pd.read_csv(BASELINE_DATASET_TEMP_0_7_PATH)
baselineDf_0 = pd.read_csv(BASELINE_DATASET_TEMP_0_PATH)
evolutionDF = pd.read_csv(EVOLUTION_DATASET_PATH)
personaDf_general = pd.read_csv(PERSONA_GENERAL_DATASET_PATH)
personaDf_math = pd.read_csv(PERSONA_MATH_DATASET_PATH)
selfInstructDf = pd.read_csv(SELF_INSTRUCT_DATASET_PATH)



In [3]:
datasetMap={BASELINE_DATASET_TEMP_0_7_PATH:baselineDf_0_7,
BASELINE_DATASET_TEMP_0_PATH:baselineDf_0,
EVOLUTION_DATASET_PATH:evolutionDF,
PERSONA_GENERAL_DATASET_PATH:personaDf_general,
PERSONA_MATH_DATASET_PATH:personaDf_math,
SELF_INSTRUCT_DATASET_PATH:selfInstructDf
}

In [4]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate


model = ChatOpenAI(
    model="gpt-5.4-mini-2026-03-17",
    reasoning={"effort": "low"}
    )

topicOptions = [
    "Prealgebra",
    "Algebra",
    "Intermediate Algebra",
    "Precalculus",
    "Geometry",
    "Differential Geometry",
    "Topology",
    "Number Theory",
    "Counting & Probability",
    "Combinatorics",
    "Graph Theory",
    "Discrete Mathematics",
    "Linear Algebra",
    "Calculus",
    "Differential Equations",
    "Real Analysis",
    "Complex Analysis",
    "Abstract Algebra",
    "Functional Analysis",
    "Optimization",
    "Probability",
    "Statistics",
    "Stochastic Processes",
    "Logic & Set Theory",
    "Mathematical Physics"
]

topicDescriptions = """
Prealgebra: arithmetic, ratios, percentages, basic equations, simple word problems.
Algebra: equations, inequalities, polynomials, factoring, expressions, functions at a standard school level.
Intermediate Algebra: advanced equation solving, sequences, series, logarithms, exponentials, radicals, conics, algebraic manipulation beyond basic algebra.
Precalculus: trigonometry, complex numbers in an elementary setting, advanced functions, identities, vectors in a non-linear-algebra setting.
Geometry: Euclidean geometry, synthetic geometry, circles, triangles, angles, area, coordinate geometry.
Differential Geometry: curves, surfaces, manifolds, curvature, geodesics, Riemannian geometry.
Topology: continuity, compactness, connectedness, homotopy, topological spaces, manifolds from a topological viewpoint.
Number Theory: integers, primes, divisibility, modular arithmetic, congruences, Diophantine equations.
Counting & Probability: benchmark-style counting/probability problems, permutations, combinations, expected value, discrete probability setups.
Combinatorics: extremal combinatorics, combinatorial constructions, invariants, generating arguments, bijections, recurrence-based counting beyond routine benchmark problems.
Graph Theory: graphs, trees, matchings, colorings, paths, flows, network structures.
Discrete Mathematics: discrete structures or mixed finite mathematics not best classified as graph theory, combinatorics, or logic.
Linear Algebra: matrices, vector spaces, eigenvalues, eigenvectors, linear transformations, determinants.
Calculus: derivatives, integrals, limits, series, multivariable calculus, standard calculus methods.
Differential Equations: ordinary or partial differential equations, dynamical systems framed primarily through ODEs/PDEs.
Real Analysis: rigorous limits, sequences of functions, metric spaces, measure, integration theory, convergence proofs.
Complex Analysis: analytic functions, contour integration, residues, holomorphic maps, complex variables.
Abstract Algebra: groups, rings, fields, modules, homomorphisms, Galois-theoretic or structural algebra.
Functional Analysis: normed spaces, Banach spaces, Hilbert spaces, operators, spectral analysis in infinite-dimensional spaces.
Optimization: linear programming, convex optimization, variational optimization, minimization or maximization as the central objective.
Probability: theoretical probability, random variables, distributions, martingales, probabilistic inequalities, stochastic reasoning beyond routine benchmark counting.
Statistics: estimation, inference, hypothesis testing, regression, statistical modeling.
Stochastic Processes: Markov chains, Brownian motion, random walks, time-indexed random systems.
Logic & Set Theory: formal logic, proof theory, model theory, set theory, cardinality, ordinals.
Mathematical Physics: mathematically formulated physics problems using advanced mathematical structures as the main lens.
""".strip()


def normalizeTopic(rawTopic: str) -> str:
    cleanedTopic = str(rawTopic).strip()
    lowerTopicOptions = {topicName.lower(): topicName for topicName in topicOptions}

    if cleanedTopic.lower() in lowerTopicOptions:
        return lowerTopicOptions[cleanedTopic.lower()]

    cleanedTopicLower = cleanedTopic.lower()
    for topicName in topicOptions:
        if topicName.lower() in cleanedTopicLower:
            return topicName

    return cleanedTopic


topicPrompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a strict mathematical topic judge.

Your task is to assign exactly one topic label to each math problem.

Use only one label from this list:
{topicOptions}

Topic guidance:
{topicDescriptions}

Decision rules:
- Return exactly one label from the provided list.
- Do not invent, merge, or rename labels.
- If multiple topics appear, choose the primary mathematical field needed to solve the problem.
- Prefer the most specific valid label when one clearly dominates.
- Use benchmark-style conventions when relevant:
  - routine olympiad or contest probability/counting problems -> Counting & Probability
  - deeper structural finite arguments, bijections, invariants, extremal methods -> Combinatorics
  - integer divisibility, primes, congruences, Diophantine structure -> Number Theory
  - synthetic or Euclidean spatial reasoning -> Geometry
  - matrices, determinants, vector spaces, eigenvalues -> Linear Algebra
  - groups, rings, fields, modules -> Abstract Algebra
  - rigorous convergence, measure, metric-space reasoning -> Real Analysis
  - contour integrals, residues, holomorphic functions -> Complex Analysis
  - ODE/PDE-centered problems -> Differential Equations
  - graphs, trees, flows, coloring -> Graph Theory
- For mixed contest problems, classify by the core idea rather than the surface wording.
- Return only the label, with no explanation, punctuation, or extra text.""",
    ),
    (
        "user",
        "Classify this math problem into one topic label:\n\n{questionText}",
    ),
])

topicChain = topicPrompt | model | StrOutputParser()


def extractTopic(questions: List[str]) -> List[str]:
    cleanedQuestions = [str(questionText) for questionText in questions]
    rawTopics = topicChain.batch(
        [
            {
                "topicOptions": ", ".join(topicOptions),
                "topicDescriptions": topicDescriptions,
                "questionText": questionText,
            }
            for questionText in cleanedQuestions
        ]
    )
    extractedTopics = [normalizeTopic(topicText) for topicText in rawTopics]
    return extractedTopics

In [ ]:
import time

for path ,df in datasetMap.items():
    df["topic"] = extractTopic(df["Question"].tolist())
    df.to_csv(path, index=False)
    time.sleep(5)



In [3]:
baselineDf['topic'].value_counts()

topic
Abstract Algebra          28
Differential Geometry      7
Algebra                    7
Number Theory              4
Lie Theory                 1
Calculus                   1
Complex Analysis           1
Differential Equations     1
Name: count, dtype: int64

### Topic Distribution And Coverage


In [11]:
#folder path to store results
RESULTS_PATH = "./results/topic/"


In [5]:
from utils.models import TopicAnalysisResult, TopicComparisonSummary, TopicCoverageSummary, TopicDatasetMetrics, TopicDistributionItem


def buildTopicDatasetMetrics(topicSeries: pd.Series, topicUniverse: List[str]) -> TopicDatasetMetrics:
    normalizedTopics = topicSeries.fillna("Unknown").astype(str).str.strip()
    normalizedTopics = normalizedTopics.where(normalizedTopics != "", "Unknown")

    topicCounts = normalizedTopics.value_counts()
    totalSamples = int(topicCounts.sum())

    topicDistribution = [
        TopicDistributionItem(
            topicName=topicName,
            count=int(topicCount),
            percentage=round((int(topicCount) / totalSamples) * 100, 4) if totalSamples else 0.0,
        )
        for topicName, topicCount in topicCounts.items()
    ]

    topicCountMap = {
        topicName: int(topicCount)
        for topicName, topicCount in topicCounts.items()
    }
    topicPercentageMap = {
        topicName: round((int(topicCount) / totalSamples) * 100, 4) if totalSamples else 0.0
        for topicName, topicCount in topicCounts.items()
    }

    observedTopics = list(topicCounts.index)
    coveredTopics = [topicName for topicName in observedTopics if topicName in topicUniverse]
    missingTopics = [topicName for topicName in topicUniverse if topicName not in topicCounts.index]
    unexpectedTopics = [topicName for topicName in observedTopics if topicName not in topicUniverse]

    dominantTopicName = observedTopics[0] if observedTopics else None
    dominantTopicCount = topicCountMap.get(dominantTopicName, 0) if dominantTopicName else 0
    dominantTopicPercentage = topicPercentageMap.get(dominantTopicName, 0.0) if dominantTopicName else 0.0

    return TopicDatasetMetrics(
        totalSamples=totalSamples,
        uniqueTopicsCount=len(observedTopics),
        observedTopics=observedTopics,
        dominantTopicName=dominantTopicName,
        dominantTopicCount=dominantTopicCount,
        dominantTopicPercentage=dominantTopicPercentage,
        topicCountMap=topicCountMap,
        topicPercentageMap=topicPercentageMap,
        topicDistribution=topicDistribution,
        topicCoverage=TopicCoverageSummary(
            coveredTopicsCount=len(coveredTopics),
            totalTopicsCount=len(topicUniverse),
            coveragePercentage=round((len(coveredTopics) / len(topicUniverse)) * 100, 4) if topicUniverse else 0.0,
            coveredTopics=coveredTopics,
            missingTopics=missingTopics,
            unexpectedTopics=unexpectedTopics,
        ),
    )



def analyzeTopicMetrics(
    baselineTopics: pd.Series,
    personaTopics: pd.Series,
    topicUniverse: List[str],
) -> TopicAnalysisResult:
    baselineMetrics = buildTopicDatasetMetrics(baselineTopics, topicUniverse)
    personaMetrics = buildTopicDatasetMetrics(personaTopics, topicUniverse)

    baselineCoveredTopics = set(baselineMetrics.topicCoverage.coveredTopics)
    personaCoveredTopics = set(personaMetrics.topicCoverage.coveredTopics)

    sharedTopics = sorted(baselineCoveredTopics & personaCoveredTopics)
    baselineOnlyTopics = sorted(baselineCoveredTopics - personaCoveredTopics)
    personaOnlyTopics = sorted(personaCoveredTopics - baselineCoveredTopics)

    return TopicAnalysisResult(
        topicUniverse=topicUniverse,
        baselineMetrics=baselineMetrics,
        personaMetrics=personaMetrics,
        comparison=TopicComparisonSummary(
            sharedTopicsCount=len(sharedTopics),
            sharedTopics=sharedTopics,
            baselineOnlyTopics=baselineOnlyTopics,
            personaOnlyTopics=personaOnlyTopics,
            coverageGapPercentagePoints=round(
                personaMetrics.topicCoverage.coveragePercentage - baselineMetrics.topicCoverage.coveragePercentage,
                4,
            ),
        ),
    )

In [ ]:
topicResult = analyzeTopicMetrics(baselineDf['topic'], personaDf['topic'], topicOptions)

In [12]:
#saving to json
import json
with open(RESULTS_PATH+'topicResult.json', 'w') as f:
    json.dump(topicResult.model_dump(), f, indent=2)

In [15]:
from typing import Any
def visualizeTopicMetrics(
    topicAnalysis: TopicAnalysisResult | dict,
    figureSize: tuple[int, int] = (20, 16),
    sortBy: str = "combinedPercentage",
) -> dict[str, Any]:
    import importlib

    try:
        plt = importlib.import_module("matplotlib.pyplot")
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError("matplotlib is required to visualize topic metrics") from exc

    resultModel: TopicAnalysisResult
    if isinstance(topicAnalysis, dict):
        resultModel = TopicAnalysisResult.model_validate(topicAnalysis)
    else:
        resultModel = topicAnalysis

    topicUniverse = list(resultModel.topicUniverse)
    baselineDistributionMap = dict(resultModel.baselineMetrics.topicPercentageMap)
    personaDistributionMap = dict(resultModel.personaMetrics.topicPercentageMap)

    extraTopics = sorted(
        (set(resultModel.baselineMetrics.observedTopics) | set(resultModel.personaMetrics.observedTopics)) - set(topicUniverse)
    )
    allTopics = topicUniverse + [topicName for topicName in extraTopics if topicName not in topicUniverse]

    distributionRows = []
    for topicName in allTopics:
        baselinePercentage = baselineDistributionMap.get(topicName, 0.0)
        personaPercentage = personaDistributionMap.get(topicName, 0.0)
        distributionRows.append(
            {
                "topicName": topicName,
                "baselinePercentage": baselinePercentage,
                "personaPercentage": personaPercentage,
                "combinedPercentage": baselinePercentage + personaPercentage,
            }
        )

    distributionDf = pd.DataFrame(distributionRows)
    if sortBy in distributionDf.columns:
        distributionDf = distributionDf.sort_values(sortBy, ascending=True)

    coverageDf = pd.DataFrame(
        [
            {
                "datasetName": "Baseline",
                "coveredTopicsCount": resultModel.baselineMetrics.topicCoverage.coveredTopicsCount,
                "missingTopicsCount": len(resultModel.baselineMetrics.topicCoverage.missingTopics),
                "unexpectedTopicsCount": len(resultModel.baselineMetrics.topicCoverage.unexpectedTopics),
            },
            {
                "datasetName": "Persona",
                "coveredTopicsCount": resultModel.personaMetrics.topicCoverage.coveredTopicsCount,
                "missingTopicsCount": len(resultModel.personaMetrics.topicCoverage.missingTopics),
                "unexpectedTopicsCount": len(resultModel.personaMetrics.topicCoverage.unexpectedTopics),
            },
        ]
    )

    overlapDf = pd.DataFrame(
        [
            {
                "groupName": "Shared",
                "topicCount": resultModel.comparison.sharedTopicsCount,
            },
            {
                "groupName": "Baseline Only",
                "topicCount": len(resultModel.comparison.baselineOnlyTopics),
            },
            {
                "groupName": "Persona Only",
                "topicCount": len(resultModel.comparison.personaOnlyTopics),
            },
        ]
    )

    dominantTopicDf = pd.DataFrame(
        [
            {
                "datasetName": "Baseline",
                "topicName": resultModel.baselineMetrics.dominantTopicName,
                "percentage": resultModel.baselineMetrics.dominantTopicPercentage,
            },
            {
                "datasetName": "Persona",
                "topicName": resultModel.personaMetrics.dominantTopicName,
                "percentage": resultModel.personaMetrics.dominantTopicPercentage,
            },
        ]
    )

    fig, axes = plt.subplots(2, 2, figsize=figureSize)

    axes[0, 0].barh(distributionDf["topicName"], distributionDf["baselinePercentage"], color="#4C78A8", alpha=0.85, label="Baseline")
    axes[0, 0].barh(distributionDf["topicName"], distributionDf["personaPercentage"], color="#F58518", alpha=0.65, label="Persona")
    axes[0, 0].set_title("Topic Distribution Comparison")
    axes[0, 0].set_xlabel("Percentage")
    axes[0, 0].set_ylabel("Topic")
    axes[0, 0].legend()

    coverageBarWidth = 0.25
    coveragePositions = list(range(len(coverageDf["datasetName"])))
    axes[0, 1].bar(
        [position - coverageBarWidth for position in coveragePositions],
        coverageDf["coveredTopicsCount"],
        width=coverageBarWidth,
        label="Covered",
        color="#54A24B",
    )
    axes[0, 1].bar(
        coveragePositions,
        coverageDf["missingTopicsCount"],
        width=coverageBarWidth,
        label="Missing",
        color="#E45756",
    )
    axes[0, 1].bar(
        [position + coverageBarWidth for position in coveragePositions],
        coverageDf["unexpectedTopicsCount"],
        width=coverageBarWidth,
        label="Unexpected",
        color="#B279A2",
    )
    axes[0, 1].set_xticks(coveragePositions)
    axes[0, 1].set_xticklabels(coverageDf["datasetName"])
    axes[0, 1].set_title("Topic Coverage Summary")
    axes[0, 1].set_ylabel("Topic Count")
    axes[0, 1].legend()

    axes[1, 0].bar(overlapDf["groupName"], overlapDf["topicCount"], color=["#72B7B2", "#4C78A8", "#F58518"])
    axes[1, 0].set_title("Covered Topic Overlap")
    axes[1, 0].set_ylabel("Topic Count")

    axes[1, 1].bar(dominantTopicDf["datasetName"], dominantTopicDf["percentage"], color=["#4C78A8", "#F58518"])
    axes[1, 1].set_title("Dominant Topic Share")
    axes[1, 1].set_ylabel("Percentage")
    for index, row in dominantTopicDf.iterrows():
        axes[1, 1].text(index, row["percentage"] + 0.5, str(row["topicName"]), ha="center", va="bottom")

    fig.suptitle("Topic Metric Visualization", fontsize=16)
    fig.tight_layout()
    plt.show()

    return {
        "figure": fig,
        "distributionDf": distributionDf,
        "coverageDf": coverageDf,
        "overlapDf": overlapDf,
        "dominantTopicDf": dominantTopicDf,
    }

In [ ]:
visualizeTopicMetrics(topicResult)

## Semantic Similarity(diversity) 

In [12]:
#creating the embedding for the questions

def lightlyNormalizeQuestion(questionText: str) -> str:
    normalizedText = "" if pd.isna(questionText) else str(questionText)
    normalizedText = unicodedata.normalize("NFKC", normalizedText)
    normalizedText = normalizedText.replace("\r\n", "\n").replace("\r", "\n")
    normalizedText = re.sub(r"^\s*Math problem:\s*", "", normalizedText, flags=re.IGNORECASE)
    normalizedText = re.sub(r"\\\(|\\\)|\\\[|\\\]", " ", normalizedText)
    normalizedText = normalizedText.replace("$", " ")
    normalizedText = re.sub(r"\\(?:left|right)\b", " ", normalizedText)
    normalizedText = re.sub(r"\\(?:,|;|:|!|quad|qquad)\b", " ", normalizedText)
    normalizedText = re.sub(r"\\(?:mathbf|mathbb|mathrm|mathcal|operatorname)\{([^{}]+)\}", r"\1", normalizedText)
    normalizedText = re.sub(r"\s+", " ", normalizedText)
    return normalizedText.strip()


def createEmbeddings(questions: List[str]) -> List[List[float]]:
    normalizedQuestions = [lightlyNormalizeQuestion(questionText) for questionText in questions]
    embeddingModel = OpenAIEmbeddings(
        model="text-embedding-3-large"
    )
    return embeddingModel.embed_documents(normalizedQuestions)

In [13]:
for path ,df in datasetMap.items():
    df["embeddings"] = createEmbeddings(df["Question"].tolist())
    df.to_csv(path, index=False)

In [ ]:
# personaDf['embeddings'] = createEmbeddings(personaDf['Question'].tolist())
# baselineDf['embeddings'] = createEmbeddings(baselineDf['Question'].tolist())

In [ ]:

# baselineDf.to_csv(BASELINE_DATASET_PATH, index=False)
# personaDf.to_csv(PERSONA_DATASET_PATH, index=False)

In [7]:
from typing import Optional, Sequence
from utils.models import (
    SimilarityAnalysisResult,
    SimilarityComparisonSummary,
    SimilarityDatasetMetrics,
    SimilarityPerQuestionMetrics,
    SimilarityTopicMetrics,
)


def _roundOrNone(value: Optional[float], decimals: int = 4) -> Optional[float]:
    if value is None:
        return None
    return round(float(value), decimals)



def _buildSimilaritySummary(pairValues: np.ndarray, nearestNeighborValues: np.ndarray, decimals: int = 4) -> dict[str, Optional[float]]:
    if pairValues.size == 0:
        return {
            "meanPairSimilarity": None,
            "medianPairSimilarity": None,
            "stdPairSimilarity": None,
            "minPairSimilarity": None,
            "maxPairSimilarity": None,
            "p10PairSimilarity": None,
            "p25PairSimilarity": None,
            "p75PairSimilarity": None,
            "p90PairSimilarity": None,
            "meanNearestNeighborSimilarity": _roundOrNone(float(np.mean(nearestNeighborValues)), decimals) if nearestNeighborValues.size else None,
            "medianNearestNeighborSimilarity": _roundOrNone(float(np.median(nearestNeighborValues)), decimals) if nearestNeighborValues.size else None,
            "maxNearestNeighborSimilarity": _roundOrNone(float(np.max(nearestNeighborValues)), decimals) if nearestNeighborValues.size else None,
            "diversityScore": None,
            "nnDiversityScore": _roundOrNone(float(1 - np.mean(nearestNeighborValues)), decimals) if nearestNeighborValues.size else None,
        }

    meanPairSimilarity = float(np.mean(pairValues))
    meanNearestNeighborSimilarity = float(np.mean(nearestNeighborValues)) if nearestNeighborValues.size else None
    return {
        "meanPairSimilarity": _roundOrNone(meanPairSimilarity, decimals),
        "medianPairSimilarity": _roundOrNone(float(np.median(pairValues)), decimals),
        "stdPairSimilarity": _roundOrNone(float(np.std(pairValues)), decimals),
        "minPairSimilarity": _roundOrNone(float(np.min(pairValues)), decimals),
        "maxPairSimilarity": _roundOrNone(float(np.max(pairValues)), decimals),
        "p10PairSimilarity": _roundOrNone(float(np.percentile(pairValues, 10)), decimals),
        "p25PairSimilarity": _roundOrNone(float(np.percentile(pairValues, 25)), decimals),
        "p75PairSimilarity": _roundOrNone(float(np.percentile(pairValues, 75)), decimals),
        "p90PairSimilarity": _roundOrNone(float(np.percentile(pairValues, 90)), decimals),
        "meanNearestNeighborSimilarity": _roundOrNone(meanNearestNeighborSimilarity, decimals),
        "medianNearestNeighborSimilarity": _roundOrNone(float(np.median(nearestNeighborValues)), decimals) if nearestNeighborValues.size else None,
        "maxNearestNeighborSimilarity": _roundOrNone(float(np.max(nearestNeighborValues)), decimals) if nearestNeighborValues.size else None,
        "diversityScore": _roundOrNone(1 - meanPairSimilarity, decimals),
        "nnDiversityScore": _roundOrNone(1 - meanNearestNeighborSimilarity, decimals) if meanNearestNeighborSimilarity is not None else None,
    }



def _computeSimilarityMatrix(embeddings: Sequence[Sequence[float]], decimals: int = 4) -> np.ndarray:
    embeddingArray = np.asarray(embeddings, dtype=float)
    if embeddingArray.ndim != 2:
        raise ValueError("Embeddings must be a 2D array-like structure.")
    if embeddingArray.shape[0] == 0:
        return np.empty((0, 0), dtype=float)

    norms = np.linalg.norm(embeddingArray, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1.0, norms)
    normalizedEmbeddings = embeddingArray / norms
    similarityMatrix = normalizedEmbeddings @ normalizedEmbeddings.T
    similarityMatrix = np.clip(similarityMatrix, -1.0, 1.0)
    return np.round(similarityMatrix, decimals)



def buildSimilarityDatasetMetrics(
    embeddings: Sequence[Sequence[float]],
    topics: Optional[Sequence[Optional[str]]] = None,
    decimals: int = 4,
) -> tuple[SimilarityDatasetMetrics, list[list[float]]]:
    similarityMatrix = _computeSimilarityMatrix(embeddings, decimals=decimals)
    totalSamples = int(similarityMatrix.shape[0])
    pairCount = int(totalSamples * (totalSamples - 1) / 2)

    upperTriangleIndices = np.triu_indices(totalSamples, k=1)
    pairValues = similarityMatrix[upperTriangleIndices] if pairCount else np.array([], dtype=float)

    perQuestionMetrics: list[SimilarityPerQuestionMetrics] = []
    nearestNeighborValues: list[float] = []
    normalizedTopics = list(topics) if topics is not None else [None] * totalSamples

    for questionIndex in range(totalSamples):
        rowValues = np.delete(similarityMatrix[questionIndex], questionIndex)
        topicName = normalizedTopics[questionIndex] if questionIndex < len(normalizedTopics) else None

        if rowValues.size == 0:
            perQuestionMetrics.append(
                SimilarityPerQuestionMetrics(
                    questionIndex=questionIndex,
                    topicName=topicName,
                )
            )
            continue

        remainingIndices = [index for index in range(totalSamples) if index != questionIndex]
        nearestNeighborPosition = int(np.argmax(rowValues))
        nearestNeighborIndex = int(remainingIndices[nearestNeighborPosition])
        nearestNeighborSimilarity = float(rowValues[nearestNeighborPosition])
        nearestNeighborValues.append(nearestNeighborSimilarity)

        perQuestionMetrics.append(
            SimilarityPerQuestionMetrics(
                questionIndex=questionIndex,
                topicName=topicName,
                meanSimilarityToOthers=_roundOrNone(float(np.mean(rowValues)), decimals),
                medianSimilarityToOthers=_roundOrNone(float(np.median(rowValues)), decimals),
                maxSimilarityToOthers=_roundOrNone(float(np.max(rowValues)), decimals),
                nearestNeighborIndex=nearestNeighborIndex,
                nearestNeighborSimilarity=_roundOrNone(nearestNeighborSimilarity, decimals),
            )
        )

    nearestNeighborArray = np.asarray(nearestNeighborValues, dtype=float) if nearestNeighborValues else np.array([], dtype=float)
    topicMetrics: list[SimilarityTopicMetrics] = []
    if topics is not None:
        topicSeries = pd.Series(normalizedTopics).fillna("Unknown").astype(str).str.strip()
        topicSeries = topicSeries.where(topicSeries != "", "Unknown")
        for topicName in topicSeries.value_counts().index.tolist():
            topicIndices = topicSeries[topicSeries == topicName].index.to_list()
            topicSampleCount = len(topicIndices)
            topicPairCount = int(topicSampleCount * (topicSampleCount - 1) / 2)

            if topicSampleCount == 1:
                topicMetrics.append(
                    SimilarityTopicMetrics(
                        topicName=topicName,
                        sampleCount=topicSampleCount,
                        pairCount=topicPairCount,
                    )
                )
                continue

            topicMatrix = similarityMatrix[np.ix_(topicIndices, topicIndices)]
            topicPairValues = topicMatrix[np.triu_indices(topicSampleCount, k=1)]
            topicNearestNeighborValues = []
            for rowIndex in range(topicSampleCount):
                topicRowValues = np.delete(topicMatrix[rowIndex], rowIndex)
                if topicRowValues.size:
                    topicNearestNeighborValues.append(float(np.max(topicRowValues)))
            topicNearestNeighborArray = np.asarray(topicNearestNeighborValues, dtype=float) if topicNearestNeighborValues else np.array([], dtype=float)
            topicSummary = _buildSimilaritySummary(topicPairValues, topicNearestNeighborArray, decimals=decimals)
            topicMetrics.append(
                SimilarityTopicMetrics(
                    topicName=topicName,
                    sampleCount=topicSampleCount,
                    pairCount=topicPairCount,
                    **topicSummary,
                )
            )

    datasetSummary = _buildSimilaritySummary(pairValues, nearestNeighborArray, decimals=decimals)
    datasetMetrics = SimilarityDatasetMetrics(
        totalSamples=totalSamples,
        pairCount=pairCount,
        perQuestionMetrics=perQuestionMetrics,
        topicMetrics=topicMetrics,
        **datasetSummary,
    )
    return datasetMetrics, similarityMatrix.tolist()



def analyzeSimilarityMetrics(
    baselineEmbeddings: Sequence[Sequence[float]],
    personaEmbeddings: Sequence[Sequence[float]],
    baselineTopics: Optional[Sequence[Optional[str]]] = None,
    personaTopics: Optional[Sequence[Optional[str]]] = None,
    decimals: int = 4,
) -> tuple[SimilarityAnalysisResult, dict[str, list[list[float]]]]:
    baselineMetrics, baselineMatrix = buildSimilarityDatasetMetrics(
        baselineEmbeddings,
        topics=baselineTopics,
        decimals=decimals,
    )
    personaMetrics, personaMatrix = buildSimilarityDatasetMetrics(
        personaEmbeddings,
        topics=personaTopics,
        decimals=decimals,
    )

    result = SimilarityAnalysisResult(
        baselineMetrics=baselineMetrics,
        personaMetrics=personaMetrics,
        comparison=SimilarityComparisonSummary(
            meanPairSimilarityGap=_roundOrNone(
                float(personaMetrics.meanPairSimilarity - baselineMetrics.meanPairSimilarity),
                decimals,
            ) if baselineMetrics.meanPairSimilarity is not None and personaMetrics.meanPairSimilarity is not None else None,
            medianPairSimilarityGap=_roundOrNone(
                float(personaMetrics.medianPairSimilarity - baselineMetrics.medianPairSimilarity),
                decimals,
            ) if baselineMetrics.medianPairSimilarity is not None and personaMetrics.medianPairSimilarity is not None else None,
            diversityScoreGap=_roundOrNone(
                float(personaMetrics.diversityScore - baselineMetrics.diversityScore),
                decimals,
            ) if baselineMetrics.diversityScore is not None and personaMetrics.diversityScore is not None else None,
            meanNearestNeighborSimilarityGap=_roundOrNone(
                float(personaMetrics.meanNearestNeighborSimilarity - baselineMetrics.meanNearestNeighborSimilarity),
                decimals,
            ) if baselineMetrics.meanNearestNeighborSimilarity is not None and personaMetrics.meanNearestNeighborSimilarity is not None else None,
            nnDiversityScoreGap=_roundOrNone(
                float(personaMetrics.nnDiversityScore - baselineMetrics.nnDiversityScore),
                decimals,
            ) if baselineMetrics.nnDiversityScore is not None and personaMetrics.nnDiversityScore is not None else None,
        ),
    )

    return result, {
        "baselineMatrix": baselineMatrix,
        "personaMatrix": personaMatrix,
    }

In [10]:
similarityResult, similarityMatrices = analyzeSimilarityMetrics(
    baselineEmbeddings=[ eval(e) for e in baselineDf["embeddings"].tolist()],
    personaEmbeddings=[eval(e) for e in personaDf["embeddings"].tolist()],
    baselineTopics=baselineDf["topic"].tolist() if "topic" in baselineDf.columns else None,
    personaTopics=personaDf["topic"].tolist() if "topic" in personaDf.columns else None,
)

baselineSimilarityMatrix = similarityMatrices["baselineMatrix"]
personaSimilarityMatrix = similarityMatrices["personaMatrix"]

similarityResult

SimilarityAnalysisResult(metricName='semantic_similarity', baselineMetrics=SimilarityDatasetMetrics(totalSamples=50, pairCount=1225, meanPairSimilarity=0.5855, medianPairSimilarity=0.56, stdPairSimilarity=0.1549, minPairSimilarity=0.2855, maxPairSimilarity=0.957, p10PairSimilarity=0.4041, p25PairSimilarity=0.4442, p75PairSimilarity=0.7167, p90PairSimilarity=0.7992, meanNearestNeighborSimilarity=0.8118, medianNearestNeighborSimilarity=0.8348, maxNearestNeighborSimilarity=0.957, diversityScore=0.4145, nnDiversityScore=0.1882, perQuestionMetrics=[SimilarityPerQuestionMetrics(questionIndex=0, topicName='Abstract Algebra', meanSimilarityToOthers=0.6358, medianSimilarityToOthers=0.6723, maxSimilarityToOthers=0.8594, nearestNeighborIndex=8, nearestNeighborSimilarity=0.8594), SimilarityPerQuestionMetrics(questionIndex=1, topicName='Number Theory', meanSimilarityToOthers=0.4767, medianSimilarityToOthers=0.4704, maxSimilarityToOthers=0.7165, nearestNeighborIndex=3, nearestNeighborSimilarity=0.71

In [12]:
# saving the similarity results
SIMILARITY_RESULTS_DIR = REPO_ROOT / "research" / "results" / "semantic_similarity"
SIMILARITY_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SIMILARITY_RESULT_PATH = SIMILARITY_RESULTS_DIR / "similarity_result.json"
BASELINE_SIMILARITY_MATRIX_PATH = SIMILARITY_RESULTS_DIR / "baseline_similarity_matrix.json"
PERSONA_SIMILARITY_MATRIX_PATH = SIMILARITY_RESULTS_DIR / "persona_similarity_matrix.json"

In [15]:
with open(SIMILARITY_RESULT_PATH, "w") as fileHandle:
    json.dump(similarityResult.model_dump(), fileHandle, indent=2)

with open(BASELINE_SIMILARITY_MATRIX_PATH, "w") as fileHandle:
    json.dump(baselineSimilarityMatrix, fileHandle, indent=2)

with open(PERSONA_SIMILARITY_MATRIX_PATH, "w") as fileHandle:
    json.dump(personaSimilarityMatrix, fileHandle, indent=2)

{
    "similarityResult": str(SIMILARITY_RESULT_PATH),
    "baselineMatrix": str(BASELINE_SIMILARITY_MATRIX_PATH),
    "personaMatrix": str(PERSONA_SIMILARITY_MATRIX_PATH),
}

{'similarityResult': '/home/zora/personal/dev/final_project/research/results/semantic_similarity/similarity_result.json',
 'baselineMatrix': '/home/zora/personal/dev/final_project/research/results/semantic_similarity/baseline_similarity_matrix.json',
 'personaMatrix': '/home/zora/personal/dev/final_project/research/results/semantic_similarity/persona_similarity_matrix.json'}